<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "192"
iso_country = "Cuba"
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2022
gdp = 633.440*1e9 

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Cuba': 

Number of trade records present in dataset: 27092
Number of partner countries present in dataset: 133
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$5428.4m
Total weight within the dataset: 5405.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,192,Cuba,440,Lithuania,Import,Sea,284700,284700 | Hydrogen peroxide; w,2847,2847 | Hydrogen peroxide; w,28,28 | Inorganic chemicals;,3428,Hydrogen peroxide; phosphides; carbides; hydri...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",4676.498775,NaN,4676.498775,NaN,0.00000,NaN,8922.292013,8470.0,5263.01237,46958.133232,['Container'],['Container'],219.476180,219.058702,219.100613,219.100613,219.289418,219.289418,229.803136,410.141985,251.879847,235.521336,236.197281,241.801385,224.921627,400.599456,266.064876,244.289248,238.488150,249.588968,220.639947,438.61989,330.598434,289.404046,267.434244,260.281419,216.437593,465.325115,382.453945,356.380539,286.416505,279.148853,212.314814,474.367120,455.050919,460.332250,308.508518,293.320233,208.109527,464.120566,442.125869,482.326452,363.318922,298.766458,204.133192,411.550589,430.352086,451.121279,333.349425,302.303802,200.094166,377.856724,383.577322,466.546555,387.466131,304.386379,3428
1,1,192,Cuba,440,Lithuania,Import,Sea,381900,381900 | Hydraulic fluids; fo,3819,3819 | Hydraulic brake flui,38,38 | Chemical products n.,3543,Lubricating preparations and preparations of a...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",9324.777150,NaN,9324.777150,kg,2139.67044,2139.67044,3139.652466,8470.0,5263.01237,16524.029768,['Container'],['Container'],77.231156,77.084251,77.098999,77.098999,77.165437,77.165437,80.865094,144.324272,88.633636,82.877263,83.115121,85.087141,79.147347,140.966365,93.625185,85.962591,83.921251,87.827502,77.640673,154.34532,116.333806,101.837972,94.107050,91.590053,76.161912,163.742583,134.581167,125.406234,100.786691,98.229287,74.711154,166.924362,160.127212,161.985652,108.560617,103.216034,73.231361,163.318716,155.579034,169.725159,127.847772,105.132498,71.832135,144.819943,151.435975,158.744416,117.301848,106.377249,70.410847,132.963458,134.976471,164.172394,136.344898,107.110084,3543


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $2139.1m (39.4%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               545169292.6  1.537863e+09     10.0     28.5
02 | Meat and edible meat  437290053.9  2.283082e+08      8.1      4.2
22 | Beverages, spirits a  216651613.7  2.755060e+08      4.0      5.1
04 | Dairy produce; birds  207143327.1  7.172646e+07      3.8      1.3
15 | Animal, vegetable or  160810810.7  1.565786e+08      3.0      2.9
07 | Vegetables and certa  107644492.5  1.726345e+08      2.0      3.2
16 | Meat, fish, crustace   93364959.1  2.107722e+07      1.7      0.4
23 | Food industries, res   88032083.2  1.753215e+08      1.6      3.2
19 | Preparations of cere   68999267.3  3.138636e+07      1.3      0.6
11 | Products of the mill   50170210.1  1.146555e+08      0.9      2.1
20 | Preparations of vege   43634594.5  3.0704

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1915.5m (35.3%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       1.387573e+09  1.723991e+09     25.6     31.9
Food & Beverages  5.264940e+08  5.159113e+08      9.7      9.5
Fishing           1.434197e+06  3.123988e+05      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1718.6m (31.7%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
OMT - Meat products n.e.c.                    496337072.6  242103701.6   
PCR - Processed rice                          265334026.5  460885204.0   
OFD - Food products n.e.c.                    251530777.6  191656676.0   
VOL - Vegetable oils and fats                 228402310.9  318607012.9   
MIL - Dairy products                          203197955.5   69649950.3   
GRO - Cereal grains n.e.c.                    121608770.9  502766532.1   
V_F - Vegetables, fruit, nuts                 114515897.2  179525556.7   
OCR - Crops n.e.c.                             27448688.3    8176613.3   
CMT - Bovine meat prods                         8327675.7    1806203.1   
PFB - Plant-based fibers                        1556966.1     833900.4   
OAP - Animal products n.e.c.                 

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 39.4% 
GTAP 31.7% 
EORA 35.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23069/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,192,Cuba,440,Lithuania,Import,Sea,284700,284700 | Hydrogen peroxide; w,2847,2847 | Hydrogen peroxide; w,28,28 | Inorganic chemicals;,3428,Hydrogen peroxide; phosphides; carbides; hydri...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",4676.498775,NaN,4676.498775,NaN,0.00000,NaN,8922.292013,8470.0,5263.01237,46958.133232,['Container'],['Container'],219.476180,219.058702,219.100613,219.100613,219.289418,219.289418,229.803136,410.141985,251.879847,235.521336,236.197281,241.801385,224.921627,400.599456,266.064876,244.289248,238.488150,249.588968,220.639947,438.61989,330.598434,289.404046,267.434244,260.281419,216.437593,465.325115,382.453945,356.380539,286.416505,279.148853,212.314814,474.367120,455.050919,460.332250,308.508518,293.320233,208.109527,464.120566,442.125869,482.326452,363.318922,298.766458,204.133192,411.550589,430.352086,451.121279,333.349425,302.303802,200.094166,377.856724,383.577322,466.546555,387.466131,304.386379,3428,NaN
1,1,192,Cuba,440,Lithuania,Import,Sea,381900,381900 | Hydraulic fluids; fo,3819,3819 | Hydraulic brake flui,38,38 | Chemical products n.,3543,Lubricating preparations and preparations of a...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",9324.777150,NaN,9324.777150,kg,2139.67044,2139.67044,3139.652466,8470.0,5263.01237,16524.029768,['Container'],['Container'],77.231156,77.084251,77.098999,77.098999,77.165437,77.165437,80.865094,144.324272,88.633636,82.877263,83.115121,85.087141,79.147347,140.966365,93.625185,85.962591,83.921251,87.827502,77.640673,154.34532,116.333806,101.837972,94.107050,91.590053,76.161912,163.742583,134.581167,125.406234,100.786691,98.229287,74.711154,166.924362,160.127212,161.985652,108.560617,103.216034,73.231361,163.318716,155.579034,169.725159,127.847772,105.132498,71.832135,144.819943,151.435975,158.744416,117.301848,106.377249,70.410847,132.963458,134.976471,164.172394,136.344898,107.110084,3543,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $1645.9m of $5428.4m (30.3%)
Volume: 2345.4 of 5405.0 million kilograms (43.4%)
Value-to-Weight: 0.7 from 1.0 US$ per kg (+-30.1%)
Average distance to market (volume-weighted): 5860.0 from 6003.3 nm (+-2.4%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    607  443628146.8  6.244711e+08      8.2     11.6
1        Meat and Prep     77  427847567.0  2.266941e+08      7.9      4.2
2  Unprocessed Cereals     24  279827992.2  1.076965e+09      5.2     19.9
3                Dairy    201  205503153.6  7.114231e+07      3.8      1.3
4        Fats and Oils    135  158572030.4  1.540464e+08      2.9      2.9
5           Vegetables    366  120554492.1  1.831490e+08      2.2      3.4
6               Fruits     79    9871544.0  8.931634e+06      0.2      0.2
7      Sugar and Honey      4     135527.0  3.885560e+04      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2.161926e+08,3.851754e+08,2.368308e+08,2.215002e+08,2.221330e+08,2.274253e+08,1.998034e+08,4.456817e+08,4.274087e+08,4.324999e+08,2.899726e+08,2.756943e+08,1.883501e+08,3.551044e+08,3.604566e+08,4.380370e+08,3.639117e+08,2.860691e+08
1,Fats and Oils,3.067721e+08,4.901005e+08,3.258225e+08,3.060772e+08,3.069024e+08,3.179413e+08,2.889066e+08,5.473093e+08,5.361929e+08,5.580795e+08,3.886558e+08,3.691450e+08,2.764325e+08,4.609108e+08,4.679805e+08,5.399486e+08,4.526404e+08,3.799237e+08
2,Fruits,1.507622e+07,2.687325e+07,1.651691e+07,1.545963e+07,1.550269e+07,1.586146e+07,1.393098e+07,3.108705e+07,2.979094e+07,3.012312e+07,2.020704e+07,1.923727e+07,1.313071e+07,2.480994e+07,2.517696e+07,3.058453e+07,2.533673e+07,1.996443e+07
3,Meat and Prep,2.273354e+08,4.057376e+08,2.491750e+08,2.329922e+08,2.336608e+08,2.392048e+08,2.100348e+08,4.692731e+08,4.501643e+08,4.553889e+08,3.051955e+08,2.901704e+08,1.979454e+08,3.737990e+08,3.794582e+08,4.615365e+08,3.833053e+08,3.011177e+08
4,Processed Cereals,2.071915e+09,3.329604e+09,2.202111e+09,2.084252e+09,2.089664e+09,2.151208e+09,1.949859e+09,3.981856e+09,3.788537e+09,3.896196e+09,2.712481e+09,2.501542e+09,1.864615e+09,3.220353e+09,3.254603e+09,3.791413e+09,3.202279e+09,2.573435e+09
5,Sugar and Honey,5.967120e+04,1.064984e+05,6.540370e+04,6.115600e+04,6.133150e+04,6.278670e+04,5.513020e+04,1.231753e+05,1.181596e+05,1.195310e+05,8.010800e+04,7.616420e+04,5.195690e+04,9.811520e+04,9.960060e+04,1.211446e+05,1.006104e+05,7.903770e+04
6,Unprocessed Cereals,3.105803e+09,5.357881e+09,3.369549e+09,3.163357e+09,3.172152e+09,3.253435e+09,2.887364e+09,6.262795e+09,5.992165e+09,6.093719e+09,4.134949e+09,3.892286e+09,2.734745e+09,5.013031e+09,5.081628e+09,6.096813e+09,5.090318e+09,4.027805e+09
7,Vegetables,3.214389e+08,5.685817e+08,3.513395e+08,3.288906e+08,3.298203e+08,3.378110e+08,2.974679e+08,6.591723e+08,6.316931e+08,6.399833e+08,4.301097e+08,4.083225e+08,2.807102e+08,5.258322e+08,5.336265e+08,6.464460e+08,5.375546e+08,4.234643e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.82,0.10,0.03,0.03,0.05,1.20,1.11,1.13,0.44,0.37,0.81,0.84,1.22,0.85,0.48
1,Fats and Oils,1.16,0.12,-0.00,0.00,0.07,1.63,1.56,1.70,0.63,0.51,1.16,1.21,1.66,1.11,0.65
2,Fruits,1.20,0.15,0.04,0.04,0.08,1.74,1.61,1.64,0.64,0.54,1.18,1.22,1.77,1.24,0.69
3,Meat and Prep,0.42,0.05,0.01,0.01,0.03,0.61,0.56,0.57,0.22,0.19,0.41,0.42,0.62,0.43,0.24
4,Processed Cereals,2.84,0.29,0.03,0.04,0.18,4.58,4.14,4.39,1.72,1.24,3.06,3.13,4.34,3.02,1.60
5,Sugar and Honey,0.35,0.04,0.01,0.01,0.02,0.50,0.47,0.48,0.18,0.16,0.34,0.35,0.51,0.36,0.20
6,Unprocessed Cereals,8.05,0.94,0.21,0.24,0.53,12.06,11.10,11.46,4.46,3.59,8.14,8.39,12.01,8.42,4.62
7,Vegetables,2.05,0.25,0.06,0.07,0.14,3.00,2.77,2.84,1.10,0.92,2.03,2.10,3.03,2.13,1.18
